### 1.Import the libraries and load the dataset

In [22]:
import os
import random
import numpy as np
import tensorflow as tf

import keras #a high-level deep-learning API that lets you build neural netwroks more easily

from keras.datasets import mnist #MNIST contains images of handwritten digits

from keras import layers, Model #functional API: build the graph as layer(x) calls instead of a fixed stack
#Conv2D => convolutional layer, looks for visual patterns in an image.
#BatchNormalization => normalizes activations between layers, stabilizes/speeds up training.
#MaxPooling2D => reduces the spatial size of the feature maps.
#GlobalAveragePooling2D => averages each feature map to one number instead of flattening (fewer params, less overfitting).
#Dropout => randomly deactivate neurons during training to reduce overfitting.
#Add => merges a skip connection with the main path (residual block).

# Reproducibility: same seed -> same weight init, same augmentation, same split.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [23]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()
print(x_train.shape, y_train.shape)

(60000, 28, 28) (60000,)


### 2.Preprocess the data

The image data cannot be fed directly into the model so we need to perform some operations and process the data to make to make it ready for our neural network. The dimension of the training data is (60000,28,28). The CNN model will require one more dimension so we reshape the matrix to shape (60000,28,28,1).

In [24]:
from sklearn.model_selection import train_test_split

x_train = x_train.reshape(x_train.shape[0], 28, 28, 1)
x_test = x_test.reshape(x_test.shape[0], 28, 28, 1)
input_shape = (28, 28, 1)

num_classes = 10

x_train = x_train.astype('float32') / 255
x_test = x_test.astype('float32') / 255

# x_test is held out untouched for final evaluation only. A proper validation
# split (used for early stopping / LR scheduling during training) comes out
# of x_train instead, so the test set never leaks into training decisions.
x_train, x_val, y_train, y_val = train_test_split(
    x_train, y_train, test_size=0.1, random_state=SEED, stratify=y_train
)

y_train = keras.utils.to_categorical(y_train, num_classes)
y_val = keras.utils.to_categorical(y_val, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)

print('x_train shape:', x_train.shape)
print(x_train.shape[0], 'train samples')
print(x_val.shape[0], 'validation samples')
print(x_test.shape[0], 'test samples')

x_train shape: (54000, 28, 28, 1)
54000 train samples
6000 validation samples
10000 test samples


### 3.Create the model

Now we will create our CNN model. Instead of a plain stack of conv layers, this one uses **residual blocks** (a small skip connection around every pair of conv layers, like ResNet) — the shortcut lets gradients flow straight through, so we can stack more layers without training getting harder. It finishes with **GlobalAveragePooling2D** instead of Flatten, which cuts the parameter count a lot and reduces overfitting on a small 28x28 input. Augmentation layers (rotation/shift/zoom) run only during training, so hand-drawn digits at inference time see the same preprocessing either way.

In [25]:
from keras.layers import RandomRotation, RandomTranslation, RandomZoom

batch_size = 128
epochs = 30

def conv_bn_relu(x, filters, kernel_size=3, strides=1):
    x = layers.Conv2D(filters, kernel_size, strides=strides, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    return layers.Activation('relu')(x)

def residual_block(x, filters):
    shortcut = x
    y = conv_bn_relu(x, filters)
    y = layers.Conv2D(filters, 3, padding='same', use_bias=False)(y)
    y = layers.BatchNormalization()(y)
    if shortcut.shape[-1] != filters:
        # 1x1 conv projects the shortcut when channel counts don't match, so it can still be added
        shortcut = layers.Conv2D(filters, 1, padding='same', use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    out = layers.Add()([shortcut, y])
    return layers.Activation('relu')(out)

inputs = layers.Input(shape=input_shape)
x = RandomRotation(0.08)(inputs)
x = RandomTranslation(0.1, 0.1)(x)
x = RandomZoom(0.1)(x)

x = conv_bn_relu(x, 32)
x = residual_block(x, 32)
x = layers.MaxPooling2D(2)(x)
x = layers.Dropout(0.25)(x)

x = residual_block(x, 64)
x = residual_block(x, 64)
x = layers.MaxPooling2D(2)(x)
x = layers.Dropout(0.25)(x)

x = residual_block(x, 128)
x = layers.GlobalAveragePooling2D()(x)

x = layers.Dense(256, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model = Model(inputs, outputs, name='mnist_resnet')
model.compile(loss=keras.losses.categorical_crossentropy, optimizer=keras.optimizers.Adam(learning_rate=1e-3), metrics=['accuracy'])
model.summary()

Model: "mnist_resnet"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 28, 28, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ random_rotation     │ (None, 28, 28, 1) │          0 │ input_layer_1[0]… │
│ (RandomRotation)    │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ random_translation  │ (None, 28, 28, 1) │          0 │ random_rotation[… │
│ (RandomTranslation) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ random_zoom         │ (None, 28, 28, 1) │          0 │ random_translati… │
│ (RandomZoom)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 28, 28,    │        288 │ random_zoom[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 28, 28,    │        128 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 28, 28,    │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 28, 28,    │      9,216 │ activation[0][0]  │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 28, 28,    │        128 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 28, 28,    │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 28, 28,    │      9,216 │ activation_1[0][… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 28, 28,    │        128 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 28, 28,    │          0 │ activation[0][0], │
│                     │ 32)               │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 28, 28,    │          0 │ add[0][0]         │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 14, 14,    │          0 │ activation_2[0][… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 14, 14,    │          0 │ max_pooling2d_1[… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 14, 14,    │     18,432 │ dropout_2[0][0] 

 Total params: 418,986 (1.60 MB)

 Trainable params: 416,874 (1.59 MB)

 Non-trainable params: 2,112 (8.25 KB)

### 4.Train the model

The model.fit() function of Keras will start the training of the model. It takes the training data, validation data, epochs, and batch size.

It takes some time to train the model. After training, we save the weights and model definition in the ‘mnist.h5’ file.

In [26]:
import matplotlib.pyplot as plt
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5),
    # Persist the best checkpoint seen during training (by val_accuracy), not
    # just whatever epoch happened to be last.
    ModelCheckpoint('mnist.h5', monitor='val_accuracy', save_best_only=True, verbose=0),
]

hist = model.fit(x_train, y_train,batch_size=batch_size,epochs=epochs,verbose=1,validation_data=(x_val, y_val),callbacks=callbacks)
print("The model has successfully trained and the best checkpoint was saved as mnist.h5")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(hist.history['accuracy'], label='train')
axes[0].plot(hist.history['val_accuracy'], label='val')
axes[0].set_title('Accuracy'); axes[0].set_xlabel('epoch'); axes[0].legend()
axes[1].plot(hist.history['loss'], label='train')
axes[1].plot(hist.history['val_loss'], label='val')
axes[1].set_title('Loss'); axes[1].set_xlabel('epoch'); axes[1].legend()
plt.tight_layout()
plt.show()

Epoch 1/30


W0000 00:00:1789491215.346975   20521 cpu_allocator_impl.cc:82] Allocation of 169344000 exceeds 10% of free system memory.


422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 552ms/step - accuracy: 0.8839 - loss: 0.3696

422/422 ━━━━━━━━━━━━━━━━━━━━ 255s 568ms/step - accuracy: 0.8839 - loss: 0.3696 - val_accuracy: 0.6828 - val_loss: 0.9315 - learning_rate: 0.0010
Epoch 2/30
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 533ms/step - accuracy: 0.9681 - loss: 0.1064

422/422 ━━━━━━━━━━━━━━━━━━━━ 230s 546ms/step - accuracy: 0.9681 - loss: 0.1064 - val_accuracy: 0.9730 - val_loss: 0.0991 - learning_rate: 0.0010
Epoch 3/30
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 531ms/step - accuracy: 0.9760 - loss: 0.0784

...

422/422 ━━━━━━━━━━━━━━━━━━━━ 269s 513ms/step - accuracy: 0.9834 - loss: 0.0561 - val_accuracy: 0.9847 - val_loss: 0.0442 - learning_rate: 0.0010
Epoch 7/30
422/422 ━━━━━━━━━━━━━━━━━━━━ 261s 512ms/step - accuracy: 0.9844 - loss: 0.0523 - val_accuracy: 0.9788 - val_loss: 0.0582 - learning_rate: 0.0010
Epoch 8/30
422/422 ━━━━━━━━━━━━━━━━━━━━ 130s 307ms/step - accuracy: 0.9860 - loss: 0.0465 - val_accuracy: 0.9755 - val_loss: 0.0853 - learning_rate: 0.0010
Epoch 9/30
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 247ms/step - accuracy: 0.9885 - loss: 0.0381

422/422 ━━━━━━━━━━━━━━━━━━━━ 106s 252ms/step - accuracy: 0.9885 - loss: 0.0381 - val_accuracy: 0.9883 - val_loss: 0.0326 - learning_rate: 5.0000e-04
Epoch 10/30
422/422 ━━━━━━━━━━━━━━━━━━━━ 113s 268ms/step - accuracy: 0.9893 - loss: 0.0351 - val_accuracy: 0.9877 - val_loss: 0.0386 - learning_rate: 5.0000e-04
Epoch 11/30
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 259ms/step - accuracy: 0.9891 - loss: 0.0345

422/422 ━━━━━━━━━━━━━━━━━━━━ 111s 264ms/step - accuracy: 0.9891 - loss: 0.0345 - val_accuracy: 0.9890 - val_loss: 0.0367 - learning_rate: 5.0000e-04
Epoch 12/30
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 248ms/step - accuracy: 0.9922 - loss: 0.0273

422/422 ━━━━━━━━━━━━━━━━━━━━ 107s 253ms/step - accuracy: 0.9922 - loss: 0.0273 - val_accuracy: 0.9928 - val_loss: 0.0251 - learning_rate: 2.5000e-04
Epoch 13/30
422/422 ━━━━━━━━━━━━━━━━━━━━ 111s 262ms/step - accuracy: 0.9922 - loss: 0.0257 - val_accuracy: 0.9895 - val_loss: 0.0326 - learning_rate: 2.5000e-04
Epoch 14/30
422/422 ━━━━━━━━━━━━━━━━━━━━ 110s 260ms/step - accuracy: 0.9917 - loss: 0.0269 - val_accuracy: 0.9920 - val_loss: 0.0241 - learning_rate: 2.5000e-04
Epoch 15/30
422/422 ━━━━━━━━━━━━━━━━━━━━ 115s 272ms/step - accuracy: 0.9915 - loss: 0.0262 - val_accuracy: 0.9918 - val_loss: 0.0275 - learning_rate: 2.5000e-04
Epoch 16/30
422/422 ━━━━━━━━━━━━━━━━━━━━ 113s 267ms/step - accuracy: 0.9923 - loss: 0.0257 - val_accuracy: 0.9910 - val_loss: 0.0282 - learning_rate: 2.5000e-04
Epoch 17/30
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 276ms/step - accuracy: 0.9927 - loss: 0.0224

422/422 ━━━━━━━━━━━━━━━━━━━━ 118s 280ms/step - accuracy: 0.9927 - loss: 0.0224 - val_accuracy: 0.9942 - val_loss: 0.0229 - learning_rate: 1.2500e-04
Epoch 18/30
422/422 ━━━━━━━━━━━━━━━━━━━━ 107s 254ms/step - accuracy: 0.9936 - loss: 0.0204 - val_accuracy: 0.9933 - val_loss: 0.0228 - learning_rate: 1.2500e-04
Epoch 19/30
422/422 ━━━━━━━━━━━━━━━━━━━━ 107s 253ms/step - accuracy: 0.9938 - loss: 0.0203 - val_accuracy: 0.9930 - val_loss: 0.0209 - learning_rate: 1.2500e-04
Epoch 20/30
422/422 ━━━━━━━━━━━━━━━━━━━━ 107s 253ms/step - accuracy: 0.9938 - loss: 0.0202 - val_accuracy: 0.9928 - val_loss: 0.0228 - learning_rate: 1.2500e-04
Epoch 21/30
422/422 ━━━━━━━━━━━━━━━━━━━━ 107s 253ms/step - accuracy: 0.9940 - loss: 0.0192 - val_accuracy: 0.9928 - val_loss: 0.0213 - learning_rate: 1.2500e-04
Epoch 22/30
422/422 ━━━━━━━━━━━━━━━━━━━━ 106s 252ms/step - accuracy: 0.9943 - loss: 0.0183 - val_accuracy: 0.9942 - val_loss: 0.0195 - learning_rate: 6.2500e-05
Epoch 23/30
422/422 ━━━━━━━━━━━━━━━━━━━━ 102s 

### 5.Evaluate the model

The 10,000-image test set was held out from both training and the validation split used for early stopping, so this is the model's first look at it — a fair estimate of real-world accuracy. Beyond the headline number, we also look at *where* it's still wrong: a confusion matrix (which digits get mixed up with which), a per-class precision/recall/F1 report, and a grid of actual misclassified images.

In [27]:
from sklearn.metrics import classification_report, confusion_matrix

score = model.evaluate(x_test, y_test, verbose=0)
print('Test loss:', score[0])
print('Test accuracy:', score[1])

y_pred_probs = model.predict(x_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

print(classification_report(y_true, y_pred, digits=3))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title('Confusion matrix')
for i in range(10):
    for j in range(10):
        color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        ax.text(j, i, cm[i, j], ha='center', va='center', color=color, fontsize=8)
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

# A handful of actual mistakes, so we can see what still trips the model up
wrong = np.where(y_pred != y_true)[0]
print(f'{len(wrong)} misclassified out of {len(y_true)} test images')
n_show = min(10, len(wrong))
if n_show:
    fig, axes = plt.subplots(1, n_show, figsize=(1.6 * n_show, 2))
    if n_show == 1:
        axes = [axes]
    for ax, idx in zip(axes, wrong[:n_show]):
        ax.imshow(x_test[idx].reshape(28, 28), cmap='gray')
        ax.set_title(f'true {y_true[idx]}\npred {y_pred[idx]}', fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

Test loss: 0.020385025069117546
Test accuracy: 0.9927999973297119
              precision    recall  f1-score   support

           0      0.995     1.000     0.997       980
           1      0.995     0.990     0.992      1135
           2      0.998     0.987     0.993      1032
           3      0.993     0.998     0.996      1010
           4      0.987     0.997     0.992       982
           5      0.998     0.990     0.994       892
           6      0.997     0.989     0.993       958
           7      0.984     0.995     0.989      1028
           8      0.985     0.998     0.991       974
           9      0.998     0.984     0.991      1009

    accuracy                          0.993     10000
   macro avg      0.993     0.993     0.993     10000
weighted avg      0.993     0.993     0.993     10000



invalid command name "124200334324800partial"
    while executing
"124200334324800partial"
    ("after" script)


72 misclassified out of 10000 test images


### Create GUI to predict digits

Draw a digit on the left. The middle panel shows the exact 28x28 image the model receives (after cropping/centering/scaling) — useful for seeing *why* a prediction went wrong. The right panel is a live bar chart of the model's confidence across all 10 digits, not just the winner.

In [28]:
from keras.models import load_model
from tkinter import *
import tkinter as tk

from PIL import Image, ImageDraw, ImageOps, ImageTk

import matplotlib
matplotlib.use('TkAgg')
from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

model = load_model('mnist.h5')

def preprocess_image(img):
    img = img.convert('L')
    #MNIST digits are white-on-black; the canvas is drawn black-on-white, so invert
    img = ImageOps.invert(img)
    arr = np.array(img)

    # Crop to the drawn strokes, then rescale/center like MNIST does (digit
    # scaled to fit a ~20px box, centered in the 28x28 frame). Without this,
    # a raw resize of a big off-center stroke looks nothing like training data.
    coords = np.argwhere(arr > 20)
    if coords.size == 0:
        return np.zeros((28, 28), dtype='float32')

    y0, x0 = coords.min(axis=0)
    y1, x1 = coords.max(axis=0) + 1
    digit = arr[y0:y1, x0:x1]

    h, w = digit.shape
    scale = 20.0 / max(h, w)
    new_h, new_w = max(1, round(h * scale)), max(1, round(w * scale))
    digit_img = Image.fromarray(digit).resize((new_w, new_h), Image.LANCZOS)

    canvas = Image.new('L', (28, 28), 0)
    offset = ((28 - new_w) // 2, (28 - new_h) // 2)
    canvas.paste(digit_img, offset)
    return np.array(canvas).astype('float32')

def predict_probs(arr28):
    x = arr28.reshape(1, 28, 28, 1) / 255.0
    return model.predict(x, verbose=0)[0]

class App(tk.Tk):
    def __init__(self):
        tk.Tk.__init__(self)
        self.title('Handwritten Digit Recognizer')

        self.x = self.y = 0
        self.canvas_size = 300
        self.preview_size = 140

        # Drawing canvas
        self.canvas = tk.Canvas(self, width=self.canvas_size, height=self.canvas_size, bg="white", cursor="cross")
        self.canvas.bind("<B1-Motion>", self.draw_lines)

        # Drawn in memory alongside the canvas, so we never depend on a screen
        # grab (ImageGrab.grab() is unreliable on Linux/X11/Wayland).
        self.image = Image.new("L", (self.canvas_size, self.canvas_size), 255)
        self.image_draw = ImageDraw.Draw(self.image)

        # What the model actually sees (28x28, cropped/centered), upscaled for display
        self.preview_label = tk.Label(self, text="Model input", compound=TOP)
        self._set_preview(np.zeros((28, 28), dtype='float32'))

        # Confidence bar chart across all 10 digits
        self.fig = Figure(figsize=(4, 3), dpi=100)
        self.ax = self.fig.add_subplot(111)
        self.chart_canvas = FigureCanvasTkAgg(self.fig, master=self)
        self._draw_barchart(np.zeros(10))

        self.result_label = tk.Label(self, text="Draw a digit", font=("Helvetica", 32))
        self.classify_btn = tk.Button(self, text="Recognise", command=self.classify_handwriting)
        self.button_clear = tk.Button(self, text="Clear", command=self.clear_all)

        # Grid structure
        self.canvas.grid(row=0, column=0, rowspan=2, padx=4, pady=4)
        self.preview_label.grid(row=0, column=1, padx=4, pady=4)
        self.chart_canvas.get_tk_widget().grid(row=0, column=2, padx=4, pady=4)
        self.result_label.grid(row=1, column=1, columnspan=2, pady=2)
        self.classify_btn.grid(row=2, column=0, pady=4)
        self.button_clear.grid(row=2, column=1, pady=4)

    def _set_preview(self, arr28):
        preview_img = Image.fromarray(arr28.astype('uint8')).resize(
            (self.preview_size, self.preview_size), Image.NEAREST
        )
        self.preview_photo = ImageTk.PhotoImage(preview_img)
        self.preview_label.configure(image=self.preview_photo)

    def _draw_barchart(self, probs):
        self.ax.clear()
        bars = self.ax.bar(range(10), probs, color='#4C72B0')
        top = int(np.argmax(probs))
        if probs[top] > 0:
            bars[top].set_color('#C44E52')
        self.ax.set_xticks(range(10))
        self.ax.set_ylim(0, 1)
        self.ax.set_title('Confidence per digit')
        self.fig.tight_layout()
        self.chart_canvas.draw()

    def clear_all(self):
        self.canvas.delete("all")
        self.image_draw.rectangle([0, 0, self.canvas_size, self.canvas_size], fill=255)
        self.result_label.configure(text="Draw a digit")
        self._set_preview(np.zeros((28, 28), dtype='float32'))
        self._draw_barchart(np.zeros(10))

    def classify_handwriting(self):
        arr28 = preprocess_image(self.image)
        probs = predict_probs(arr28)
        digit = int(np.argmax(probs))
        confidence = float(probs[digit])

        self.result_label.configure(text=f'{digit}  ({confidence*100:.1f}%)')
        self._set_preview(arr28)
        self._draw_barchart(probs)

    def draw_lines(self, event):
        self.x = event.x
        self.y = event.y
        r = 8
        self.canvas.create_oval(self.x-r, self.y-r, self.x+r, self.y+r, fill='black')
        self.image_draw.ellipse([self.x-r, self.y-r, self.x+r, self.y+r], fill=0)

app = App()
mainloop()